In [9]:
# 1. Install python3.10 and venv support
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y
# 2. Create an isolated virtual environment
!python3.10 -m venv /content/venv
# 3. Upgrade pip inside the virtual environment
!/content/venv/bin/python -m pip install --upgrade pip
# 4. Install the required serving pins
!/content/venv/bin/python -m pip install \
    "vllm==0.6.*" \
    "transformers==4.46.*" \
    "accelerate==1.1.*" \
    "httpx==0.27.*" \
    "openai==1.54.*"
print("Virtual environment ready with vLLM installed!")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [10]:
import subprocess, sys, os, signal, time, urllib.request, urllib.error, csv, threading

VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = ["/content/venv/bin/python", "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
}

def build_cmd(args: dict) -> list:
    cmd = ["/content/venv/bin/python", "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    proc = subprocess.Popen(
        cmd, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True
    )
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server()

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    return False

healthy = wait_for_health()

GPU_SAMPLES = "/content/gpu_samples.csv"
_sampler = {"thread": None, "stop": None}

def _sample_loop(stop_event, path, interval_s):
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["t", "util_gpu", "mem_used_mib"])
        t0 = time.time()
        while not stop_event.is_set():
            out = subprocess.run(
                [
                    "nvidia-smi",
                    "--query-gpu=utilization.gpu,memory.used",
                    "--format=csv,noheader,nounits"
                ],
                capture_output=True,
                text=True
            ).stdout.strip()
            parts = [p.strip() for p in out.split(",")]
            if len(parts) == 2:
                w.writerow([round(time.time() - t0, 2), parts[0], parts[1]])
                fh.flush()
            stop_event.wait(interval_s)

def start_sampler(path=GPU_SAMPLES, interval_s=2):
    if _sampler["thread"] and _sampler["thread"].is_alive():
        print("sampler already running; not starting a second one")
        return
    stop = threading.Event()
    th = threading.Thread(
        target=_sample_loop,
        args=(stop, path, interval_s),
        daemon=True
    )
    th.start()
    _sampler["thread"], _sampler["stop"] = th, stop
    print(f"sampler started -> {path} (every {interval_s}s)")

def stop_sampler():
    if _sampler["stop"]:
        _sampler["stop"].set()
    if _sampler["thread"]:
        _sampler["thread"].join(timeout=5)
    _sampler["thread"], _sampler["stop"] = None, None
    print("sampler stopped")

def read_util_mean(path=GPU_SAMPLES):
    vals = []
    with open(path) as fh:
        for row in csv.DictReader(fh):
            try:
                vals.append(float(row["util_gpu"]))
            except (KeyError, ValueError):
                pass
    return sum(vals) / len(vals) if vals else 0.0

def shutdown_server(proc=None, port=PORT):
    try:
        proc = server if proc is None else proc
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        print(f"sent SIGTERM to process group of pid {proc.pid}")
    except (ProcessLookupError, NameError):
        print("no server process to kill")
    time.sleep(3)
    try:
        with urllib.request.urlopen(
            f"http://localhost:{port}/v1/models",
            timeout=2
        ):
            print(f"WARNING: port {port} still answering; something is still up")
    except (urllib.error.URLError, ConnectionError, OSError):
        print(f"port {port} is free")

launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000
server pid 7602, logging to /content/server.log
TIMED OUT after 300s waiting for http://localhost:8000/v1/models
last 30 log lines:
/content/venv/bin/python: Error while finding module specification for 'vllm.entrypoints.openai.api_server' (ModuleNotFoundError: No module named 'vllm')



In [12]:
import os, sys, time, signal, subprocess, urllib.request, urllib.error

RECOVERY_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
RECOVERY_PORT = 8000
RECOVERY_LOG = "/content/server.log"

_R_TRANSFORMERS = "4.46.*"
_R_ACCELERATE = "1.1.*"
_R_NEED_AWQ = False
_R_VLLM = "0.6.*"; _R_HTTPX = "0.27.*"; _R_OPENAI = "1.54.*"

RECOVERY_ARGS = {
    "--model": RECOVERY_MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(RECOVERY_PORT),
}

subprocess.run(["pkill", "-f", "vllm.entrypoints.openai.api_server"], check=False)
time.sleep(2)

subprocess.run(["/content/venv/bin/python", "-m", "pip", "install", "-q",
                f"vllm=={_R_VLLM}", f"transformers=={_R_TRANSFORMERS}",
                f"accelerate=={_R_ACCELERATE}", f"httpx=={_R_HTTPX}",
                f"openai=={_R_OPENAI}"], check=True)

_r_cmd = ["/content/venv/bin/python", "-m", "vllm.entrypoints.openai.api_server"]
for k, v in RECOVERY_ARGS.items():
    _r_cmd += [k] if v is None else [k, str(v)]
_r_logf = open(RECOVERY_LOG, "wb")
server = subprocess.Popen(_r_cmd, stdout=_r_logf, stderr=subprocess.STDOUT, start_new_session=True)

_deadline = time.time() + 300
while time.time() < _deadline:
    try:
        with urllib.request.urlopen(f"http://localhost:{RECOVERY_PORT}/v1/models", timeout=5) as r:
            if r.status == 200:
                print("RECOVERED: server healthy. continue from your last step.")
                break
    except (urllib.error.URLError, ConnectionError, OSError):
        pass
    time.sleep(3)

CalledProcessError: Command '['/content/venv/bin/python', '-m', 'pip', 'install', '-q', 'vllm==0.6.*', 'transformers==4.46.*', 'accelerate==1.1.*', 'httpx==0.27.*', 'openai==1.54.*']' returned non-zero exit status 1.

In [13]:
subprocess.run([
    "/content/venv/bin/python", "-m", "pip", "install",
    f"vllm=={_R_VLLM}",
    f"transformers=={_R_TRANSFORMERS}",
    f"accelerate=={_R_ACCELERATE}",
    f"httpx=={_R_HTTPX}",
    f"openai=={_R_OPENAI}"
], check=True)

CalledProcessError: Command '['/content/venv/bin/python', '-m', 'pip', 'install', 'vllm==0.6.*', 'transformers==4.46.*', 'accelerate==1.1.*', 'httpx==0.27.*', 'openai==1.54.*']' returned non-zero exit status 1.

In [14]:
import subprocess

result = subprocess.run(
    [
        "/content/venv/bin/python", "-m", "pip", "install",
        "vllm==0.6.*",
        "transformers==4.46.*",
        "accelerate==1.1.*",
        "httpx==0.27.*",
        "openai==1.54.*"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)


/content/venv/bin/python: No module named pip



In [15]:
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "ensurepip", "--upgrade"
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "--upgrade", "pip"
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "vllm==0.6.*",
    "transformers==4.46.*",
    "accelerate==1.1.*",
    "httpx==0.27.*",
    "openai==1.54.*"
], check=True)

CalledProcessError: Command '['/usr/bin/python3', '-m', 'ensurepip', '--upgrade']' returned non-zero exit status 1.

In [16]:
import os, subprocess, sys

if not os.path.exists("/usr/bin/python3.11"):
    subprocess.run([
        "apt-get", "update", "-qq"
    ], check=True)
    subprocess.run([
        "apt-get", "install", "-y", "-qq", "python3.11", "python3.11-venv"
    ], check=True)

if not os.path.exists("/content/venv/bin/python"):
    subprocess.run([
        "/usr/bin/python3.11", "-m", "venv", "/content/venv"
    ], check=True)

subprocess.run([
    "/content/venv/bin/python", "-m", "ensurepip", "--upgrade"
], check=True)

subprocess.run([
    "/content/venv/bin/python", "-m", "pip", "install", "--upgrade", "pip"
], check=True)

subprocess.run([
    "/content/venv/bin/python", "-m", "pip", "install",
    "vllm==0.6.*",
    "transformers==4.46.*",
    "accelerate==1.1.*",
    "httpx==0.27.*",
    "openai==1.54.*"
], check=True)

print("DONE")
print("Python:", subprocess.check_output([
    "/content/venv/bin/python", "--version"
]).decode().strip())

CalledProcessError: Command '['/content/venv/bin/python', '-m', 'ensurepip', '--upgrade']' returned non-zero exit status 1.

In [17]:
import os, subprocess

subprocess.run(["rm", "-rf", "/content/venv"], check=True)

if not os.path.exists("/usr/bin/python3.11"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run([
        "apt-get", "install", "-y", "-qq",
        "python3.11", "python3.11-venv"
    ], check=True)

subprocess.run([
    "/usr/bin/python3.11", "-m", "venv", "/content/venv"
], check=True)

subprocess.run([
    "/content/venv/bin/python", "-m", "pip",
    "install", "--upgrade", "pip"
], check=True)

print("Python:", subprocess.check_output([
    "/content/venv/bin/python", "--version"
]).decode().strip())

print("pip:", subprocess.check_output([
    "/content/venv/bin/python", "-m", "pip", "--version"
]).decode().strip())

Python: Python 3.11.15
pip: pip 26.2.1 from /content/venv/lib/python3.11/site-packages/pip (python 3.11)


In [18]:
import subprocess

subprocess.run([
    "/content/venv/bin/python", "-m", "pip", "install",
    "vllm==0.6.*",
    "transformers==4.46.*",
    "accelerate==1.1.*",
    "httpx==0.27.*",
    "openai==1.54.*"
], check=True)

CompletedProcess(args=['/content/venv/bin/python', '-m', 'pip', 'install', 'vllm==0.6.*', 'transformers==4.46.*', 'accelerate==1.1.*', 'httpx==0.27.*', 'openai==1.54.*'], returncode=0)

In [19]:
import subprocess

subprocess.run([
    "/content/venv/bin/python", "-m", "pip", "show", "vllm"
])

CompletedProcess(args=['/content/venv/bin/python', '-m', 'pip', 'show', 'vllm'], returncode=0)

In [20]:
import subprocess

RECOVERY_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
RECOVERY_PORT = 8000
RECOVERY_LOG = "/content/server.log"

RECOVERY_ARGS = {
    "--model": RECOVERY_MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(RECOVERY_PORT),
}

subprocess.run(
    ["pkill", "-f", "vllm.entrypoints.openai.api_server"],
    check=False
)

_r_cmd = [
    "/content/venv/bin/python",
    "-m",
    "vllm.entrypoints.openai.api_server"
]

for k, v in RECOVERY_ARGS.items():
    _r_cmd += [k] if v is None else [k, str(v)]

_r_logf = open(RECOVERY_LOG, "wb")

server = subprocess.Popen(
    _r_cmd,
    stdout=_r_logf,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print(f"server started: PID {server.pid}")
print(f"log: {RECOVERY_LOG}")

server started: PID 12609
log: /content/server.log


In [21]:
import time
import urllib.request
import urllib.error

deadline = time.time() + 300

while time.time() < deadline:
    try:
        with urllib.request.urlopen(
            "http://localhost:8000/v1/models",
            timeout=5
        ) as r:
            if r.status == 200:
                print("SERVER HEALTHY ✅")
                break
    except (urllib.error.URLError, ConnectionError, OSError):
        pass

    time.sleep(3)
else:
    print("SERVER TIMED OUT ❌")
    with open("/content/server.log", "r", errors="replace") as f:
        print("".join(f.readlines()[-30:]))

SERVER HEALTHY ✅


In [22]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")

r = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[{"role": "user", "content": "In one sentence, what is a GPU?"}],
)

print(r.choices[0].message.content)

A GPU, or Graphics Processing Unit, is a specialized processor designed to accelerate computations involved in rendering graphics and video content on electronic devices.


In [23]:
import json

sample_data = {
    "_note": "SAMPLE baseline from the reference T4 run 2026-08-31.",
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "dtype": "fp16",
    "ttft_s": {
        "128": 0.0371,
        "512": 0.0647,
        "2048": 0.3123
    },
    "tpot_s": 0.0341,
    "batch": {
        "1": 34.0,
        "4": 49.8,
        "8": 96.6
    }
}

with open("baselines.sample.json", "w") as f:
    json.dump(sample_data, f, indent=2)

print("baselines.sample.json created successfully!")

baselines.sample.json created successfully!


In [24]:
raw_code = '''
import asyncio
import time
import httpx

FIXED_PROMPTS = [
    "In one sentence, what is a GPU?",
    "List three reasons decode is memory-bound.",
    "Explain the KV cache to a new ops engineer in two sentences.",
    "What does continuous batching change versus static batching?",
    "Give a one-line definition of tokens per second.",
    "Why does a longer prompt increase time to first token?",
    "Name two things quantisation trades away for smaller memory.",
    "Summarise what an inference server does in three short bullets.",
]

QUEUE = [32, 32, 32, 256] * 6
MAX_TOKENS = 128
WARMUP = 4

async def _one_request(client, base_url, model, prompt, max_tokens=MAX_TOKENS):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stream": False,
    }
    r = await client.post(f"{base_url}/chat/completions", json=payload)
    r.raise_for_status()
    body = r.json()
    usage = body.get("usage", {})
    ct = usage.get("completion_tokens")
    if ct is None:
        ct = len(body["choices"][0]["message"]["content"].split())
    return ct

async def _run_level(client, base_url, model, prompts, concurrency, total_requests):
    sem = asyncio.Semaphore(concurrency)
    counts = []

    async def guarded(prompt, max_tokens):
        async with sem:
            return await _one_request(client, base_url, model, prompt, max_tokens)

    tasks = [asyncio.create_task(guarded(prompts[i % len(prompts)],
                                        QUEUE[i % len(QUEUE)]))
             for i in range(total_requests)]
    t0 = time.time()
    for coro in asyncio.as_completed(tasks):
        counts.append(await coro)
    dt = time.time() - t0
    total_tokens = sum(counts)
    return {
        "concurrency": concurrency,
        "requests": total_requests,
        "tokens_per_s": round(total_tokens / dt, 1),
        "wall_s": round(dt, 3),
    }

async def run_sweep(base_url, model, prompts=FIXED_PROMPTS,
                    concurrencies=(1, 4, 8), requests_per_level=24):
    results = []
    async with httpx.AsyncClient(timeout=120.0) as client:
        await asyncio.gather(*[
            _one_request(client, base_url, model, prompts[i % len(prompts)])
            for i in range(WARMUP)
        ])
        for c in concurrencies:
            level = await _run_level(client, base_url, model, prompts, c,
                                     requests_per_level)
            print("level:", level)
            results.append(level)
    return results
'''

cleaned_code = raw_code.replace('\xa0', ' ')
with open("ab_client.py", "w") as f:
    f.write(cleaned_code)

from ab_client import run_sweep, FIXED_PROMPTS

vllm_measured = await run_sweep(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompts=FIXED_PROMPTS,
    concurrencies=[1, 4, 8],
)

print("Sweep completed successfully!")

level: {'concurrency': 1, 'requests': 24, 'tokens_per_s': 58.6, 'wall_s': 23.686}
level: {'concurrency': 4, 'requests': 24, 'tokens_per_s': 164.3, 'wall_s': 8.455}
level: {'concurrency': 8, 'requests': 24, 'tokens_per_s': 232.6, 'wall_s': 5.972}
Sweep completed successfully!


In [25]:
import json

baseline = json.load(open("baselines.sample.json"))

vllm_by_c = {x["concurrency"]: x["tokens_per_s"] for x in vllm_measured}
base_by_c = {int(k): v for k, v in baseline["batch"].items()}

speedup = {
    c: round(vllm_by_c[c] / base_by_c[c], 2)
    for c in vllm_by_c
    if c in base_by_c
}

report = {
    "baseline": base_by_c,
    "vllm": vllm_by_c,
    "speedup_by_concurrency": speedup,
    "predicted_speedup": 1.5,
}

with open("ab_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Generated ab_report.json successfully:\n")
print(json.dumps(report, indent=2))

Generated ab_report.json successfully:

{
  "baseline": {
    "1": 34.0,
    "4": 49.8,
    "8": 96.6
  },
  "vllm": {
    "1": 58.6,
    "4": 164.3,
    "8": 232.6
  },
  "speedup_by_concurrency": {
    "1": 1.72,
    "4": 3.3,
    "8": 2.41
  },
  "predicted_speedup": 1.5
}


In [26]:
static_scaling = base_by_c[8] / base_by_c[1]
vllm_scaling   = vllm_by_c[8] / vllm_by_c[1]

print(f"static batching scales {static_scaling:.2f}x, vLLM scales {vllm_scaling:.2f}x")
print(f"continuous batching is worth {vllm_scaling / static_scaling:.2f}x of scaling")

static batching scales 2.84x, vLLM scales 3.97x
continuous batching is worth 1.40x of scaling


In [27]:
import json
import os

def verify_report():
    report_path = "ab_report.json"
    assert os.path.exists(report_path), f"Error: {report_path} not found!"

    with open(report_path, "r") as f:
        report = json.load(f)

    # 1. Check schema / required keys
    required_keys = ["baseline", "vllm", "speedup_by_concurrency", "predicted_speedup"]
    for k in required_keys:
        assert k in report, f"Schema Error: Missing key '{k}' in ab_report.json"

    # 2. Check that vLLM concurrency-8 throughput beats Monday's batch-8 baseline
    vllm_8 = report["vllm"].get("8") or report["vllm"].get(8)
    base_8 = report["baseline"].get("8") or report["baseline"].get(8)

    assert vllm_8 is not None and base_8 is not None, "Concurrency 8 metrics missing from report."
    assert vllm_8 > base_8, f"Performance check failed: vLLM concurrency-8 ({vllm_8}) must exceed baseline ({base_8})."

    # 3. Verify speedup fields were computed
    speedups = report["speedup_by_concurrency"]
    assert len(speedups) > 0, "Speedup fields were not computed."
    for level, ratio in speedups.items():
        assert isinstance(ratio, (int, float)) and ratio > 0, f"Invalid speedup ratio for level {level}: {ratio}"

    print("GREEN CHECK: PASS")

verify_report()

GREEN CHECK: PASS


In [28]:
# Green-check verifier for Lab W3D3 (engine swap).
# Paste this as the last cell of your day-3 notebook and run it. It reads
# ab_report.json and checks the schema, that vLLM's concurrency-8 throughput
# beats Monday's batch-8 baseline, and that the speedup fields were computed.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    path = "ab_report.json"
    if not os.path.exists(path):
        fail(f"{path} not found; write it in Cell 5")
    try:
        with open(path) as fh:
            report = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"{path} is not valid JSON: {exc}")

    for key in ("baseline", "vllm", "speedup_by_concurrency"):
        if key not in report:
            fail(f"{path} missing key: {key}")

    baseline = report["baseline"]
    vllm = report["vllm"]
    speedup = report["speedup_by_concurrency"]

    if not isinstance(baseline, dict) or not baseline:
        fail("baseline must be a non-empty object (from Monday's baselines.json)")
    if not isinstance(vllm, dict) or not vllm:
        fail("vllm must be a non-empty object of measured throughput")
    if not isinstance(speedup, dict) or not speedup:
        fail("speedup_by_concurrency must be a non-empty object")

    # keys may be strings or ints depending on how the report was built; normalise
    def get_c(d, c):
        for k, v in d.items():
            if str(k) == str(c):
                return v
        return None

    base8 = get_c(baseline, 8)
    vllm8 = get_c(vllm, 8)
    if base8 is None:
        fail("baseline has no concurrency-8 (batch-8) number")
    if vllm8 is None:
        fail("vllm has no concurrency-8 number")
    if not isinstance(base8, (int, float)) or not isinstance(vllm8, (int, float)):
        fail("concurrency-8 throughput values must be numbers")

    # the headline claim of the day
    if not vllm8 > base8:
        fail(f"vllm concurrency-8 throughput ({vllm8}) not above baseline "
             f"batch-8 ({base8}); the engine swap should win here")

    # speedup fields must be computed (present and numeric for at least c=8)
    s8 = get_c(speedup, 8)
    if s8 is None or not isinstance(s8, (int, float)):
        fail("speedup_by_concurrency has no numeric value at concurrency 8")
    # sanity: the reported speedup should match vllm8/base8 within rounding
    expected = vllm8 / base8
    if abs(s8 - expected) > 0.1:
        fail(f"speedup at 8 ({s8}) does not match vllm/baseline "
             f"({expected:.2f}); recompute it")

    print(f"baseline batch-8: {base8}, vllm concurrency-8: {vllm8}")
    print(f"speedup at 8: {s8}x")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


baseline batch-8: 96.6, vllm concurrency-8: 232.6
speedup at 8: 2.41x
GREEN CHECK: PASS


In [29]:
from google.colab import files

files.download("ab_client.py")
files.download("ab_report.json")
files.download("baselines.sample.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>